In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
import os

In [2]:
dataset = load_dataset("databricks/databricks-dolly-15k")
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})

In [3]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})
['instruction', 'context', 'response', 'category']
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [4]:
ejemplo = dataset["train"][0]

for k, v in ejemplo.items():
    print(f"\n{k.upper()}")
    print(v)


INSTRUCTION
When did Virgin Australia start operating?

CONTEXT
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

RESPONSE
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.

CATEGORY
closed_qa


In [5]:
messages = [{"role": "user", "content": ejemplo["instruction"] + (f"\n\nContext: {ejemplo["context"]}" if ejemplo["context"] else "")},
            {"role": "assistant", "content": ejemplo["response"]}] 

In [6]:
messages

[{'role': 'user',
  'content': "When did Virgin Australia start operating?\n\nContext: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."},
 {'role': 'assistant',
  'content': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.'}]

In [7]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",
                                          token=os.getenv("HF_TOKEN"))

In [8]:
tokenizer

TokenizersBackend(name_or_path='meta-llama/Llama-3.2-3B-Instruct', vocab_size=128000, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>'}, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_special_token_2|>", rstrip=False, lstrip=False, single_word=False, normalize

In [9]:
print(tokenizer.chat_template)

{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- 

In [10]:
texto = tokenizer.apply_chat_template(messages, tokenize=False)
print(texto)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 05 Aug 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

When did Virgin Australia start operating?

Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|eot_id|>


In [11]:
def format_example(example):
    messages = [{"role": "user", "content": example["instruction"] + (f"\n\nContext: {example["context"]}" if example["context"] else "")},
                {"role":"assistant", "content":example["response"]}]
    return {"messages": messages}

In [12]:
dataset = dataset.map(format_example)

In [17]:
def apply_chat_template(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False
    )}

In [18]:
tokenized_dataset = dataset.map(apply_chat_template)

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [24]:
tokenized_dataset["train"].column_names

['instruction', 'context', 'response', 'category', 'messages', 'text']